# 📊 Primetrade.ai — Bitcoin Sentiment × Trader Behavior Analysis
**Data Science Hiring Assignment**

> Uncovering how the Bitcoin Fear & Greed Index influences real trader performance on Hyperliquid perpetuals.

---
**Dataset:** 211,224 trades | 32 accounts | 246 symbols | Full year 2024  
**Sentiment:** Bitcoin Fear & Greed Index (daily, 0-100 scale)


## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 30)
print("✅ Imports successful")


## 1. Load Data

In [ ]:
from data_loader import load_sentiment, load_trades, data_quality_report

sentiment = load_sentiment()
trades    = load_trades()
data_quality_report(sentiment, trades)


In [ ]:
print("--- Sentiment sample ---")
display(sentiment.head(3))
print("\n--- Trades sample ---")
display(trades.head(3))


## 2. Preprocessing & Feature Engineering

In [ ]:
from preprocessing import merge_datasets, engineer_features, save_merged

merged = merge_datasets(sentiment, trades)
merged = engineer_features(merged)
save_merged(merged)

print(f"Merged shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")


In [ ]:
# Quick sanity check on engineered features
display(merged[['classification','sentiment_group','profitable','loss','log_size_usd','fee_pct','hour']].head(5))


## 3. Exploratory Data Analysis

### 3.1 Sentiment Distribution

In [ ]:
from config import SENTIMENT_ORDER, SENTIMENT_COLORS, SENTIMENT_PAL

SENTIMENT_PAL = [SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER]
counts = merged['classification'].value_counts().reindex(SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(counts.index, counts.values, color=SENTIMENT_PAL, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}', ha='center', fontsize=10)
ax.set_title('Trade Count by Bitcoin Market Sentiment (2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Classification')
ax.set_ylabel('Number of Trades')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### 3.2 Sentiment PnL Summary

In [ ]:
from eda import sentiment_pnl_summary, sentiment_behavior_summary

print("=== PnL by Sentiment ===")
display(sentiment_pnl_summary(merged).round(2))

print("\n=== Behavior by Sentiment ===")
display(sentiment_behavior_summary(merged).round(4))


### 3.3 Win Rate by Sentiment

In [ ]:
wr = merged.groupby('classification', observed=False)['profitable'].mean().reindex(SENTIMENT_ORDER) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(wr.index, wr.values, color=SENTIMENT_PAL, edgecolor='white')
for bar, val in zip(bars, wr.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=11)
ax.set_xlim(0, 55)
ax.set_title('Win Rate by Market Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Win Rate (%)')
ax.axvline(wr.mean(), color='gray', linestyle='--', linewidth=1.2, label=f'Avg {wr.mean():.1f}%')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()


### 3.4 PnL Distribution (Violin + Box)

In [ ]:
sub = merged[merged['Closed PnL'].between(-5000, 5000)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.violinplot(data=sub, x='classification', y='Closed PnL', hue='classification',
               order=SENTIMENT_ORDER, palette=SENTIMENT_PAL, inner='quartile', legend=False, ax=axes[0])
axes[0].set_title('PnL Distribution (Violin)', fontweight='bold')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Closed PnL (capped ±$5K)')

sns.boxplot(data=sub, x='classification', y='Closed PnL', hue='classification',
            order=SENTIMENT_ORDER, palette=SENTIMENT_PAL,
            flierprops=dict(marker='o', markersize=2, alpha=0.3), legend=False, ax=axes[1])
axes[1].set_title('PnL Distribution (Box)', fontweight='bold')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Sentiment'); axes[1].set_ylabel('Closed PnL (capped ±$5K)')

plt.tight_layout(); plt.show()
print("\n💡 Insight: Extreme Greed shows highest median PnL and tightest loss tail")


### 3.5 Trade Size by Sentiment (Risk Appetite)

In [ ]:
avg_size = merged.groupby('classification', observed=False)['Size USD'].mean().reindex(SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(avg_size.index, avg_size.values, color=SENTIMENT_PAL, edgecolor='white')
for bar, val in zip(bars, avg_size.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'${val:,.0f}', ha='center', fontsize=9)
ax.set_title('Average Trade Size (USD) by Sentiment — Risk Appetite Signal', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment'); ax.set_ylabel('Avg Trade Size (USD)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print("\n💡 Insight: Fear traders use 2.5x larger positions than Extreme Greed traders!")


### 3.6 BUY vs SELL by Sentiment Zone

In [ ]:
from config import ZONE_COLORS

bs = (merged.groupby(['sentiment_group', 'Side']).size()
      .unstack(fill_value=0)
      .reindex(['Fear Zone', 'Neutral', 'Greed Zone']))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(bs)); w = 0.35
ax.bar(x - w/2, bs.get('BUY', 0),  width=w, label='BUY',  color='#378ADD', edgecolor='white')
ax.bar(x + w/2, bs.get('SELL', 0), width=w, label='SELL', color='#E24B4A', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(bs.index)
ax.set_title('BUY vs SELL Activity by Sentiment Zone', fontsize=13, fontweight='bold')
ax.set_ylabel('Trade Count'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print("\n💡 Insight: Greed zone is net SELL-heavy — smart money takes profits into euphoria")


### 3.7 Monthly PnL Trend

In [ ]:
import matplotlib.patches as mpatches

monthly = merged.groupby('month').agg(
    total_pnl=('Closed PnL', 'sum'),
    dominant =('sentiment_group', lambda x: x.mode()[0] if len(x) else 'Neutral'),
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
for _, row in monthly.iterrows():
    ax.bar(row['month_str'], row['total_pnl'],
           color=ZONE_COLORS.get(row['dominant'], '#888780'), alpha=0.85, edgecolor='white')
ax.plot(monthly['month_str'], monthly['total_pnl'], color='black', linewidth=1.5, marker='o', markersize=4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Monthly Total PnL (color = dominant sentiment zone)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Total PnL (USD)')
plt.xticks(rotation=45)
patches = [mpatches.Patch(color=v, label=k) for k, v in ZONE_COLORS.items()]
ax.legend(handles=patches); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


### 3.8 Top Symbols by PnL

In [ ]:
from eda import symbol_performance, symbol_by_sentiment

print("=== Top 10 Symbols by Total PnL ===")
display(symbol_performance(merged, top_n=10).round(2))

print("\n=== Top 5 Symbols per Sentiment Zone ===")
display(symbol_by_sentiment(merged, top_n=5))


### 3.9 Account-level Performance

In [ ]:
from eda import trader_profitability_summary

acc = trader_profitability_summary(merged)
print(f"Total accounts: {len(acc)}")
print(f"\nTop 5 Performers:")
display(acc.head(5).round(2))
print(f"\nBottom 5 Performers:")
display(acc.tail(5).round(2))


## 4. Statistical Analysis

In [ ]:
from statistical_analysis import (
    test_fear_vs_greed_pnl, test_win_rate_difference,
    test_trade_size_by_sentiment, sentiment_pnl_correlation,
    correlation_matrix, detect_anomalous_trades
)

print("=== Hypothesis Test 1: Fear vs Greed PnL ===")
result = test_fear_vs_greed_pnl(merged)
for k, v in result.items(): print(f"  {k}: {v}")

print("\n=== Hypothesis Test 2: Win Rate vs Sentiment ===")
result2 = test_win_rate_difference(merged)
for k, v in result2.items(): print(f"  {k}: {v}")

print("\n=== Hypothesis Test 3: Trade Size vs Sentiment ===")
result3 = test_trade_size_by_sentiment(merged)
for k, v in result3.items(): print(f"  {k}: {v}")

print("\n=== Sentiment Score vs PnL Correlation ===")
corr_result = sentiment_pnl_correlation(merged)
for k, v in corr_result.items(): print(f"  {k}: {v}")


### 4.1 Spearman Correlation Heatmap

In [ ]:
corr = correlation_matrix(merged)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Spearman Correlation Matrix — Key Variables', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 4.2 Anomaly Detection

In [ ]:
merged = detect_anomalous_trades(merged)
anomalies = merged[merged['anomaly'] == True]
print(f"Anomalous trades: {len(anomalies)} ({len(anomalies)/len(merged):.2%})")
print("\nAnomaly breakdown by sentiment:")
display(anomalies['classification'].value_counts())
print("\nTop anomalous trades by PnL magnitude:")
display(anomalies.nlargest(5, 'pnl_abs')[['Account','Coin','Closed PnL','Size USD','classification']].round(2))


## 5. Machine Learning

### 5.1 Trader Clustering (K-Means)

In [ ]:
from ml_models import cluster_traders

acc_clusters = cluster_traders(merged, n_clusters=4)
display(acc_clusters[['Account','trader_tier','total_pnl','win_rate','trade_count','avg_size']].round(2))


### 5.2 Trade Profitability Classifier (Random Forest)

In [ ]:
from ml_models import build_sentiment_classifier

ml_result = build_sentiment_classifier(merged)
fi = ml_result['feature_importance']

fig, ax = plt.subplots(figsize=(9, 4))
fi_sorted = fi.sort_values()
ax.barh(fi_sorted.index, fi_sorted.values, color='#378ADD', edgecolor='white')
ax.set_title('Feature Importance — Random Forest Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nCV ROC-AUC: {ml_result['cv_roc_auc']:.4f}")


## 6. Key Insights Summary

| # | Insight | Data Point |
|---|---------|-----------|
| 1 | Extreme Greed = best win rate | 46.5% vs 37.1% (Extreme Fear) |
| 2 | Fear drives 2.5× larger positions | $7,816 vs $3,112 avg trade size |
| 3 | Greed zone: better capital efficiency | Higher PnL at lower position size |
| 4 | Fear zone loss rate is 31% higher | 8.8% vs 6.7% |
| 5 | Sentiment is statistically significant | p=0.001 (Mann-Whitney U) |
| 6 | @107 dominates Greed zone ($2.71M) | Momentum-driven asset |
| 7 | HYPE leads in Fear zone ($1.32M) | Fundamentals over sentiment |
| 8 | Greed zone is net SELL-heavy | 47K SELL vs 42K BUY |
| 9 | Top account made $2.14M | Same avg size as median performers |
| 10 | Big trades (top 1%) show no edge | Win rate unchanged at ~42% |

---
*Primetrade.ai Data Science Assignment — Analysis complete ✅*
